Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## A ReAct agent

- The loop is : the model asks for a tool, the tool runs, the result goes back
- It repeats until the model answers without asking for anything
- Finishing is the absence of a tool call

Builds a chain, then a tool, then an agent that decides when to use it.

### Exercise Assemble the chain: `Prompt` → `LLM` → `StrOutputParser`

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
# 1) Prepare a PromptTemplate with a single variable 'query'
prompt = PromptTemplate.from_template("Answer concisely: {query}")  # TODO: change the text if you like
# 2) Prepare the LLM (we don't run any queries in this task)
llm = make_llm()
# 3) Text parser
parser = StrOutputParser()
# 4) Assemble the chain using the | operators
chain = ____ | ____ | ____  # TODO
assert chain is not None, "The chain was not created."
```

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template("Answer concisely: {query}")

llm = make_llm()

parser = StrOutputParser()

chain = prompt | llm | parser
assert chain is not None, "The chain was not created."

# Actually run the chain and print the result
result = chain.invoke({"query": "In one sentence: what is a backorder?"})
print(result)

### Exercise 2 Tool: decorate a function with @tool

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# Goal: define a simple function and register it as a LangChain tool.
from typing import Annotated
from langchain_core.tools import tool

# 1) Create a function that returns the length of the text
@tool
def text_length(s: Annotated[str, "Input text"]) -> int:
    """Returns the number of characters in the text."""
    return ____ # TODO

# 2) Use the tool directly
res = text_length.invoke({"s": "LangChain"})
print("Length:", res)
assert isinstance(res, int), "The tool should return an int."
```

In [ ]:
# Goal: define a simple function and register it as a LangChain tool.
from typing import Annotated
from langchain_core.tools import tool

# 1) Create a function that returns the length of the text
@tool
def text_length(s: Annotated[str, "Input text"]) -> int:
    """Returns the number of characters in the text."""
    return len(s)          # <- was ____

# 2) Use the tool directly
res = text_length.invoke({"s": "LangChain"})
print("Length:", res)
assert isinstance(res, int), "The tool should return an int."

### Exercise ReAct agent: create a sketch of an agent that uses tools

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
# Goal: Prepare a minimal sketch of a ReAct agent with a single tool.
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = make_llm()

# List of tools available to the agent:
tools = ____          # TODO: e.g. [text_length]

# Create the ReAct agent
agent = create_agent(
    ____,             # TODO: the model
    ____,             # TODO: the list of tools
)

response = agent.invoke({"messages": [("user", "Measure the length of this text: konstantynopolitańczykowianeczka")]})
print(response["messages"][-1].content)
```

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = make_llm()

tools = [text_length]          # the tool from the previous exercise

agent = create_agent(llm, tools)

response = agent.invoke(
    {"messages": [("user", "Measure the length of this text: konstantynopolitańczykowianeczka")]}
)

# Show the full exchange of messages
for msg in response["messages"]:
    msg.pretty_print()

### Try a question needing two tools

- Ask something that cannot be answered with one call, and watch it go round twice
- Then ask something needing no tool at all. A good agent answers directly
  rather than reaching for one anyway